[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/doubled_algebra/doubled_algebra.ipynb)

# The Doubled Algebra

This example builds a larger algebra in which every linear map on the multivectors of a space is itself a multivector. Giving each basis vector of the space a partner of the opposite square doubles the algebra, and in the doubled algebra adding a basis vector to a multivector as a factor, removing one, and every combination of the two are products: a map is an element, and applying maps one after another is multiplying them.

Each basis vector `x` and its partner `u` make two null vectors, `(x + u) / 2`, which adds `x` to a blade as a wedge does, and `(x - u) / 2`, which removes it as a contraction does. The original multivectors sit in a left ideal of the doubled algebra, so left multiplication by any element is a map on them, `State <- State`, and with n basis vectors the doubled algebra has exactly as many components, 4ⁿ, as there are such maps: each map is one multivector, recovered by a trace pairing and one solve, and the composition of maps is the geometric product of their multivectors. A product of an adding and a removing vector is a dyad that acts on every factor of a blade, so its exponential carries areas and volumes along with vectors.

In the notation of fermionic Fock space the adding and removing vectors read as creation and annihilation operators $a_i^\dagger$ and $a_i$, with $\{a_i, a_j^\dagger\} = \delta_{ij}$, and the statement as $\mathrm{Cl}(n, n) \cong \mathrm{End}(\Lambda \mathbb R^n)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
from functools import reduce
from operator import mul

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure
from examples.math.doubled_algebra import render
from examples.math.doubled_algebra.core import grid_samples

np.set_printoptions(precision=3, suppress=True)

In [ ]:
# Two positive and two negative basis vectors give Cl(2,2), with sixteen blades.
ga = Algebra("x+y+u-v-")
context = NumpyContext(ga)
mv = context.multivector
Full = ga.gatype.full()                                     # Full+
# The states have the four components of a planar multivector: 1, x, y, xy.
State = Full.derive.within("x y")                           # Full+
Vector = ga.gatype.vector().derive.within("x y")            # Vector+
n_state_components = len(State.output_subspace)

## 1. Adding and removing a basis vector

Wedging a basis vector into a blade adds it as a factor, while contraction removes it, with the sign set by its position in the blade. Each operation squares to zero. Pairing a positive and a negative basis vector produces two null vectors whose products have exactly those properties. Their mixed anticommutator is the scalar product of the vectors they add and remove, one for `x` with itself.

In [ ]:
basis = stack([mv.x, mv.y])                                 # [2] Vector+
partners = stack([mv.u, mv.v])                              # [2] Vector+
creators = (basis + partners) / 2                           # [2] Vector+
removers = (basis - partners) / 2                           # [2] Vector+

# Each family consists of null vectors. The mixed pair has a nonzero pairing.
create_x, create_y = creators                               # [] Vector+, Vector+
remove_x, remove_y = removers                               # [] Vector+, Vector+
pairing_x = 2 * remove_x.anticommutator(create_x)           # [] Scalar+

print("Add x")
print(create_x.coefficient_table())
print("Remove x")
print(remove_x.coefficient_table())
print("Their mixed anticommutator")
print(pairing_x.coefficient_table())

## 2. Multiplication as a map on states

An empty state is killed by either remover. Multiplying it by creators supplies the other states, all within a four-dimensional left ideal: a space closed under multiplication from the left. The projector embeds the four independent coefficients into this ideal; reading them out undoes that embedding. Multiplying between these two steps gives a unary extensor, `State <- State`.

In [ ]:
# remove_x * create_x keeps states without x; the two such projectors commute.
projector = reduce(mul, removers * creators)                # [] Even+
embedder = State * projector                                # [] Full+ <- State+
# Keep the independent components. Each projector contributes a factor of one half.
state_readout = (n_state_components * Full).cast(State)     # [] State+ <- Full+
representations = state_readout(Full * embedder)           # [] State+ <- (Full+, State+)

# Each multiplying element defines a map from states to states.
insertion = representations(create_x)                       # [] State+ <- State+
removal = representations(remove_x)                         # [] State+ <- State+

print("Add x: State <- State")
print(insertion.coefficient_table())
print("Remove x: State <- State")
print(removal.coefficient_table())

In [ ]:
state = 1 + 2 * mv.x - mv.y + 3 * mv.xy                     # [] State+

# create_x + remove_x is mv.x: ordinary geometric multiplication adds and removes.
left_multiplication = representations(create_x + remove_x)  # [] State+ <- State+
product = left_multiplication(state)                        # [] State+

print("State")
print(state.coefficient_table())
print("Multiplication by x: State <- State")
print(left_multiplication.coefficient_table())
print("Product")
print(product.coefficient_table())

## 3. A shear from one mixed product

Removing `y` and then adding `x` sends `y` to `x` and kills `x`: a dyad. On a blade it replaces each factor in turn, which is the product rule defining a derivation. Exponentiating the same element therefore shears the vectors and carries their wedge with them. The square becomes a parallelogram while its area stays fixed.

In [ ]:
amount = 0.8
extent, n_lines = 1.5, 9
boundary = mv(Vector, [[-1, -1], [1, -1], [1, 1], [-1, 1]])  # [corners] Vector+
grid = grid_samples(basis, extent, n_lines)                  # [lines, endpoints] Vector+

In [ ]:
# The right factor acts first: remove y, then add x.
generator = create_x * remove_y                             # [] Even+
# This generator squares to zero, so its exponential terminates after the linear term.
transformation = representations((generator * amount).exp())   # [] State+ <- State+
# The shear keeps grades, so restricted to vectors it is a map from vectors to vectors.
vector_shear = transformation(Vector).cast(Vector)          # [] Vector+ <- Vector+
grid_image = vector_shear(grid)                             # [lines, endpoints] Vector+
boundary_image = vector_shear(boundary)                     # [corners] Vector+
basis_image = vector_shear(basis)                           # [2] Vector+
area_image = transformation(mv.xy)                          # [] State+

print("Generator: State <- State")
print(representations(generator).coefficient_table())
print("Image of the area")
print(area_image.coefficient_table())
save_figure(render.draw_shear(grid, grid_image, boundary, boundary_image, basis_image), "doubled_algebra_shear");

The grey grid and dashed square show the reference shape; the blue grid and orange boundary show the shear. The arrows are the images of `x` and `y`: `x` stays fixed while `y` tilts towards it.

The figure draws each vector at the tip of its coefficients, and so reads the shear's action on vectors as an action on the places of the plane. That clashes with the plane-based reading, in which places are antivectors and a vector is a hyperplane, in the plane a line through the origin. The construction has no room for places: its states are the exterior algebra the wedge builds from vectors, while the antivectors of Cl(2,2) have grade three and involve the partners `u` and `v`. Taken instead as complements of vectors within the plane, places move the other way round under the same exponential: the points of the `y` axis stay where they are, and those of the `x` axis tilt away from `y`. The drawn parallelogram is the image of the vectors; whether the clash is a limit of the orientation paradigm or of a construction built on vectors is an open question.

## 4. Products beyond vector maps

Removing and reinserting a basis vector keeps precisely the blades containing it. Summing over the basis therefore counts a blade's grade. Products of two creators or two removers do something different: they add or remove a pair of factors. The exchange below swaps scalar and area components while killing both vector components, so its action cannot be inferred from its restriction to vectors.

In [ ]:
# Each term counts whether its basis vector occurs in the state.
grade_count = (creators * removers).sum(axis=0)             # [] Even+
# Add both basis vectors, or remove them in the reverse order.
exchange = create_x * create_y + remove_y * remove_x        # [] Even+
exchanged_state = representations(exchange)(state)          # [] State+

print("Count basis vectors: State <- State")
print(representations(grade_count).coefficient_table())
print("Exchange scalar and area: State <- State")
print(representations(exchange).coefficient_table())
print("Exchanged state")
print(exchanged_state.coefficient_table())

## 5. Every unary map as a multivector

The sixteen independent elements of this algebra represent all maps on four state components. A map is read back into a multivector by tracing it against the representations of all sixteen blades and solving against the scalar product. The two printed maps below agree; the multivector between them contains the same information, and composing maps corresponds to multiplying their encoded multivectors.

In [ ]:
measured = 2 * mv.x + mv.y                                  # [] Vector+
returned = 1 + mv.x - mv.xy                                 # [] State+
coupling = 0.3

# A scalar measurement with a mixed-grade return, plus insertion of y.
state_map = returned * measured.scalar_product(State) + coupling * (mv.y ^ State)   # [] State+ <- State+
# The trace measures the prescribed state map against the algebra's representation.
pairing = Full.scalar_product(Full)                         # [] Scalar+ <- (Full+, Full+)
multivector = pairing.solve(representations(Full, state_map).trace(0, 2) / n_state_components)   # [] Full+
recovered = representations(multivector)                    # [] State+ <- State+

print("Prescribed map: State <- State")
print(state_map.coefficient_table())
print("Encoded multivector")
print(multivector.coefficient_table())
print("Recovered map: State <- State")
print(recovered.coefficient_table())

## 6. The same construction in four dimensions

Four basis vectors and their four partners give Cl(4,4): 256 components acting on the 16 components of a four-dimensional multivector. The construction is unchanged. A single mixed product still has only four blade coefficients; acting on `yzt` replaces `y` with `x`, without touching the other factors.

In [ ]:
ga4 = Algebra("x+y+z+t+u-v-w-s-")
context4 = NumpyContext(ga4)
mv4 = context4.multivector
Full4 = ga4.gatype.full()
State4 = Full4.derive.within("x y z t")
n_state_components4 = len(State4.output_subspace)

In [ ]:
basis4 = stack([mv4.x, mv4.y, mv4.z, mv4.t])                # [4] Vector+
partners4 = stack([mv4.u, mv4.v, mv4.w, mv4.s])             # [4] Vector+
creators4 = (basis4 + partners4) / 2                        # [4] Vector+
removers4 = (basis4 - partners4) / 2                        # [4] Vector+

projector4 = reduce(mul, removers4 * creators4)             # [] Even+
embedder4 = State4 * projector4                             # [] Full+ <- State+
state_readout4 = (n_state_components4 * Full4).cast(State4)   # [] State+ <- Full+
representations4 = state_readout4(Full4 * embedder4)       # [] State+ <- (Full+, State+)

# The same dyad acts on every factor of a higher-grade blade.
create_x4 = (mv4.x + mv4.u) / 2                             # [] Vector+
remove_y4 = (mv4.y - mv4.v) / 2                             # [] Vector+
generator4 = create_x4 * remove_y4                          # [] Even+
state4 = mv4.yzt                                            # [] State+
image4 = representations4(generator4)(state4)               # [] State+

print("Generator")
print(generator4.coefficient_table())
print("State")
print(state4.coefficient_table())
print("Image")
print(image4.coefficient_table())

In [ ]:
# checks
# Adding and removing x are the wedge and the contraction; the shear keeps the area; the encoded
# multivector gives back the prescribed map; and composing two maps is multiplying their multivectors.
wedge = mv.x ^ State                                        # [] State+ <- State+
contraction = mv.x.left_contraction(State)                  # [] State+ <- State+
np.testing.assert_allclose((insertion - wedge).kernel, 0, atol=1e-12)
np.testing.assert_allclose((removal - contraction).kernel, 0, atol=1e-12)
np.testing.assert_allclose((area_image - mv.xy).kernel, 0, atol=1e-12)
np.testing.assert_allclose((recovered - state_map).kernel, 0, atol=1e-12)
composed = representations(create_x * remove_y)
np.testing.assert_allclose((composed - insertion(representations(remove_y))).kernel, 0, atol=1e-12)

This wedge-and-contraction representation is the spinor construction for a split algebra; the exterior states themselves are its spinors. Gualtieri develops it in [Generalized complex geometry, §1.3](https://www.math.toronto.edu/mgualt/gcg/gcg.pdf#page=5), alongside the transformations that mix the two null families.